# Auditoría de integridad de las cuatro pestañas — Agosto 2026

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Auditoria_Integridad_Clon_Agosto_Colab.ipynb)

**Solo lectura**, sin ZIP, sin escrituras a Sheets, sin modificar ETL o front. Verifica encabezados, registros, claves, referencias y categorías operativas de `Procesos`, `Ofertas`, `Resultados` y `OrdenesCompra`.

**Criterio de cobertura:** una OC fuera del corte mensual es una limitación temporal reconocida y no se trata como incidencia que deba subsanarse consultando otros meses. Las relaciones OC–COT no se pueden reconstruir por igualdad directa de `codigo_proceso` cuando se trata de códigos distintos.

In [ ]:
!pip -q install gspread google-auth
from google.colab import auth,files
from google.auth import default
import gspread,collections,json,datetime,csv
auth.authenticate_user()
creds,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets.readonly'])
gc=gspread.authorize(creds)
book=gc.open_by_key('1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE')
expected={
 'Procesos':['codigo','nombre','estado','convocatoria','fecha_publicacion','fecha_cierre','monto_disponible_clp','organismo_comprador','region','total_ofertas_recibidas','motivo_seleccion','url_detalle','fecha_sincronizacion','categoria','subcategoria','modalidad'],
 'Ofertas':['codigo','proveedor','monto_ofertado_clp','fecha_oferta','fuente','id_cotizacion','seleccionado'],
 'Resultados':['codigo','adjudicatario','monto_adjudicado_clp','fecha_resultado','motivo_seleccion','fuente','validado','id_cotizacion_ganadora','monto_oferta_seleccionada_clp','estado_verificacion_oc'],
 'OrdenesCompra':['codigo_oc','codigo_proceso','fecha_creacion','fecha_aceptacion','estado','proveedor','monto_total_oc_clp','tipo_oc','cantidad_lineas','total_lineas_neto']
}
print('Conexión solo lectura establecida')

In [ ]:
def read_sheet(name,batch=5000):
 ws=book.worksheet(name)
 head=expected[name]
 assert ws.row_values(1)==head,f'Cabecera distinta en {name}'
 rows=[]
 for start in range(2,ws.row_count+1,batch):
  end=min(ws.row_count,start+batch-1)
  data=ws.get(f'A{start}:{chr(64+len(head))}{end}',value_render_option='UNFORMATTED_VALUE')
  for r in data:
   if not r or not any(str(v).strip() for v in r):continue
   vals=[str(x).strip() if x is not None else '' for x in r]
   vals+=['']*(len(head)-len(vals))
   rows.append(dict(zip(head,vals[:len(head)])))
  if start%25000<batch:print(name,'leídas:',len(rows))
 return rows
data={name:read_sheet(name) for name in expected}
print('Filas recuperadas:',{k:len(v) for k,v in data.items()})

In [ ]:
P,O,R,C=[data[n] for n in ('Procesos','Ofertas','Resultados','OrdenesCompra')]
cp=collections.Counter(x['codigo'].upper() for x in P if x['codigo'])
co=collections.Counter(x['id_cotizacion'] for x in O if x['id_cotizacion'])
cr=collections.Counter(x['codigo'].upper() for x in R if x['codigo'])
cc=collections.Counter(x['codigo_oc'].upper() for x in C if x['codigo_oc'])
proc=set(cp)
oferta_por_id={x['id_cotizacion']:x for x in O if x['id_cotizacion']}
issues=[]
def count_missing(rows,key):return sum(not r[key] for r in rows)
def duplicates(ct):return sum(v-1 for v in ct.values() if v>1)
counters={
 'Procesos':{'filas':len(P),'claves_vacias':count_missing(P,'codigo'),'duplicados':duplicates(cp)},
 'Ofertas':{'filas':len(O),'claves_vacias':count_missing(O,'id_cotizacion'),'duplicados':duplicates(co),
            'sin_proceso_referenciado':sum(x['codigo'].upper() not in proc for x in O)},
 'Resultados':{'filas':len(R),'claves_vacias':count_missing(R,'codigo'),'duplicados':duplicates(cr),
               'sin_proceso_referenciado':sum(x['codigo'].upper() not in proc for x in R),
               'sin_oferta_ganadora_referenciada':sum(x['id_cotizacion_ganadora'] not in oferta_por_id for x in R if x['id_cotizacion_ganadora']),
               'seleccion_proveedor_distinto':sum(x['id_cotizacion_ganadora'] in oferta_por_id and x['adjudicatario']!=oferta_por_id[x['id_cotizacion_ganadora']]['proveedor'] for x in R)},
 'OrdenesCompra':{'filas':len(C),'claves_vacias':count_missing(C,'codigo_oc'),'duplicados':duplicates(cc),
   'codigo_proceso_vacio':count_missing(C,'codigo_proceso')}
}
estados=collections.Counter(x['estado_verificacion_oc'] for x in R)
resumen={'fecha_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'destino':'clon solo lectura','controles':counters,'estados_resultados':dict(estados),
 'cobertura_temporal':'No se cuentan OC fuera del corte mensual como incidencias reparables',
 'advertencia':'La ausencia de coincidencia codigo_proceso entre OC y COT no implica falta de vínculo oficial'}
print(json.dumps(resumen,ensure_ascii=False,indent=2))

In [ ]:
p='/content/auditoria_integridad_clon_agosto_2026.json'
with open(p,'w',encoding='utf-8') as f:json.dump(resumen,f,ensure_ascii=False,indent=2)
files.download(p)
print('Auditoría terminada; ninguna hoja modificada')